# Loan Default Risk Predictor

**Goal:** Predict whether a loan applicant is likely to default, using their financial profile.

This project extends concepts from the *TATA GenAI Powered Data Analytics Job Simulation* (exploratory data analysis, risk profiling, and delinquency prediction) into an independent, end-to-end machine learning workflow.

**Dataset:** Simulated loan applicant data (8,000 records) with realistic financial relationships (credit score, income, debt-to-income ratio, etc.) modeled on patterns seen in real-world credit risk datasets.

**Tools used:** Python, pandas, scikit-learn, matplotlib, seaborn

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve

%matplotlib inline

## 2. Load the Data

In [ ]:
df = pd.read_csv('loan_data.csv')
print('Shape:', df.shape)
df.head()

## 3. Explore the Data (EDA)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()  # check for missing values

In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='default', data=df)
plt.title('Default vs Non-Default Count')
plt.xlabel('Default (0 = No, 1 = Yes)')
plt.show()

print('Default rate:', df['default'].mean())

### Key observation
About 30% of applicants in this dataset defaulted. This is a realistic rate for a higher-risk loan portfolio (real consumer lending portfolios often range 5-25% depending on the borrower segment).

In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(x='default', y='credit_score', data=df)
plt.title('Credit Score by Default Status')
plt.xlabel('Default (0 = No, 1 = Yes)')
plt.show()

In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(x='default', y='debt_to_income_ratio', data=df)
plt.title('Debt-to-Income Ratio by Default Status')
plt.xlabel('Default (0 = No, 1 = Yes)')
plt.show()

In [ ]:
plt.figure(figsize=(9,7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.show()

### Key observation
Credit score and debt-to-income ratio show the clearest visual separation between defaulters and non-defaulters, suggesting they will be strong predictors in the model.

## 4. Prepare Data for Modeling

In [ ]:
X = df.drop(columns=['default'])
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {len(X_train)}')
print(f'Testing samples: {len(X_test)}')
print(f'Default rate in training set: {y_train.mean():.2%}')
print(f'Default rate in test set: {y_test.mean():.2%}')

## 5. Train Model 1 — Logistic Regression (baseline)

Logistic Regression is a simple, interpretable model — a good baseline before trying something more complex.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)

log_preds = log_model.predict(X_test_scaled)
log_probs = log_model.predict_proba(X_test_scaled)[:, 1]

log_accuracy = accuracy_score(y_test, log_preds)
log_auc = roc_auc_score(y_test, log_probs)

print(f'Accuracy: {log_accuracy:.2%}')
print(f'ROC-AUC Score: {log_auc:.3f}')
print(classification_report(y_test, log_preds))

## 6. Train Model 2 — Random Forest (more powerful)

Random Forest can capture more complex, non-linear patterns than Logistic Regression.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, rf_preds)
rf_auc = roc_auc_score(y_test, rf_probs)

print(f'Accuracy: {rf_accuracy:.2%}')
print(f'ROC-AUC Score: {rf_auc:.3f}')
print(classification_report(y_test, rf_preds))

## 7. Evaluate Results Visually

In [ ]:
cm = confusion_matrix(y_test, rf_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
fpr_log, tpr_log, _ = roc_curve(y_test, log_probs)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)

plt.figure(figsize=(7,6))
plt.plot(fpr_log, tpr_log, label=f'Logistic Regression (AUC={log_auc:.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={rf_auc:.3f})')
plt.plot([0,1], [0,1], 'k--', label='Random guess baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()

## 8. Which Factors Matter Most?

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8,5))
importances.plot(kind='barh', color='steelblue')
plt.title('Feature Importance - What Predicts Default Risk Most')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.show()

print(importances)

## 9. Conclusion

- Both models perform meaningfully better than random guessing (AUC > 0.7).
- **Credit score** is by far the strongest predictor of default risk, followed by **loan amount**, **annual income**, and **debt-to-income ratio**.
- Random Forest slightly outperforms Logistic Regression, suggesting some non-linear relationships in the data.
- **Business implication:** A lender could use this model to flag high-risk applicants for manual review before approval, similar to the collections-strategy use case explored in the TATA GenAI Data Analytics simulation.

### Possible next steps
- Try additional models (XGBoost, Gradient Boosting)
- Tune hyperparameters with GridSearchCV
- Test on a real-world Kaggle credit risk dataset for comparison